<a href="https://colab.research.google.com/github/TrungTan369/Machine-Learning-Assignment/blob/main/notebooks/ex3_imageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
print("Hello world from colab web font-end")

Hello world from colab web font-end


In [4]:
# Cell 1: Tải dataset từ Kaggle (kagglehub)
# Cài đặt và tải dataset INRIA (chạy trên Colab)
%pip install kagglehub -q
import kagglehub, os
print("Downloading INRIA Person from Kaggle...")
dataset_path = kagglehub.dataset_download("jcoral02/inriaperson")
print("Done. dataset_path =", dataset_path)

100%|██████████| 582M/582M [00:09<00:00, 64.2MB/s] 

Extracting files...


Done. dataset_path = /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


In [5]:
# Cell 2: Machine Learning Traditional - EDA, feature-extract (pretrained), save
import zipfile, shutil
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt

# nếu zip -> giải nén
if isinstance(dataset_path, str) and dataset_path.endswith('.zip'):
    extract_dir = '/content/inriaperson'
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(dataset_path,'r') as z: z.extractall(extract_dir)
    dataset_root = extract_dir
else:
    dataset_root = dataset_path

def find_image_root(root):
    p = Path(root)
    for child in p.iterdir():
        if child.is_dir() and any(child.glob('**/*.jpg')): return str(p)
    for sub in p.rglob('*'):
        if sub.is_dir() and any(sub.glob('*.jpg')): return str(sub)
    return str(p)

dataset_for_loader = find_image_root(dataset_root)
print("Dataset root:", dataset_for_loader)

# Cấu hình
image_size=(224,224); batch_size=32; model_name='resnet50'; pooling='avg'

from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk

# Load + EDA
X_raw, y_raw, class_names = load_and_preprocess_data(dataset_for_loader, image_size=image_size, batch_size=batch_size, shuffle=True)
print("Images:", X_raw.shape, "Labels:", y_raw.shape, "Classes:", class_names)
unique, counts = np.unique(y_raw, return_counts=True)
for u,c in zip(unique,counts): print(f"Class {u} ({class_names[u]}): {c}")
print("Pixel dtype/min/max/mean:", X_raw.dtype, X_raw.min(), X_raw.max(), X_raw.mean())

# show samples
to_show = min(len(class_names),6)
fig,axs = plt.subplots(1,to_show,figsize=(3*to_show,3))
for i in range(to_show):
    idx = (y_raw==unique[i]).nonzero()[0][0]
    axs[i].imshow(X_raw[idx].astype('uint8')); axs[i].axis('off'); axs[i].set_title(class_names[unique[i]])
plt.show()

# Extract features (pretrained)
X_features = extract_features_pretrained(X_raw, model_name=model_name, batch_size=batch_size, pooling=pooling)
print("Features shape:", X_features.shape)

# Save
prefix = f"inria_{model_name}_{image_size[0]}"
x_path, y_path = save_features_to_disk(X_features, y_raw, prefix)
print("Saved:", x_path, y_path)

Dataset root: /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


ModuleNotFoundError: No module named 'modules'

In [ ]:
# Cell 3: Deep Learning - Transfer learning end-to-end (train/val/test)
import tensorflow as tf
from pathlib import Path

# Tạo tf.data từ thư mục (dùng dataset_for_loader)
img_size = image_size
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='training', seed=42)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='validation', seed=42)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Build model (transfer learning)
base = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(*img_size,3))
base.trainable = False
inputs = tf.keras.Input(shape=(*img_size,3))
x = tf.keras.applications.resnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(train_ds.class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

# Train head
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# (Optional) Fine-tune some base layers
base.trainable = True
for layer in base.layers[:-20]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)

# Evaluate and save model
model.evaluate(val_ds)
model.save('models/inria_transfer_resnet50')
print("Saved transfer model to models/inria_transfer_resnet50")